# Official Vendi pooled image diversity

This notebook computes image diversity with the official [`vendi_score`](https://github.com/vertaix/Vendi-Score) package and its predefined Inception-v3 image embedding utilities.

The generated score is the primary result. The optional real-image score provides a diversity reference. All images are recursively pooled across class directories.


## 1. Vendi-only dependencies

The official package provides an image extra containing its predefined image utilities:

```bash
pip install "vendi_score[images]"
```

Set `INSTALL_DEPS = True` once to install it into this notebook kernel. This notebook does not need Google CMMD, Scenic, DGM-Eval, Transformers, or SciPy pins from the other metrics.


In [ ]:
from __future__ import annotations

import gc
import importlib.util
import json
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

from IPython.display import Markdown, display

print("Python:", sys.version.split()[0])
print("vendi_score installed:", importlib.util.find_spec("vendi_score") is not None)


In [ ]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "vendi_score[images]"
    ], check=True)
    print("Vendi image dependencies installed. Restart the kernel before continuing.")
else:
    print("Dependency installation skipped. Set INSTALL_DEPS = True if needed.")


## 2. Evaluation configuration

The official image utility uses pretrained torchvision Inception-v3 pool embeddings. `MAX_IMAGES = None` evaluates every recursively discovered image.


In [ ]:
SAMPLES_DIR = Path("/path/to/samples")
REAL_DIR = Path("/path/to/real")

DEVICE = "cuda"  # cuda, cuda:0, cpu, or mps
BATCH_SIZE = 32
MAX_IMAGES = None  # None means all images.
COMPUTE_REAL_REFERENCE = True

OUTPUT_PATH = Path("official_vendi.json")
RUN_VENDI = False


In [ ]:
IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}


def discover_images(root: Path, maximum: int | None) -> list[Path]:
    if not root.is_dir():
        return []
    paths = sorted(
        path.resolve()
        for path in root.rglob("*")
        if path.is_file() and path.suffix.casefold() in IMAGE_EXTENSIONS
    )
    return paths if maximum is None else paths[:maximum]


sample_paths = discover_images(SAMPLES_DIR.expanduser(), MAX_IMAGES)
real_paths = discover_images(REAL_DIR.expanduser(), MAX_IMAGES) if COMPUTE_REAL_REFERENCE else []
checks = {
    "sample_images": len(sample_paths),
    "real_images": len(real_paths),
    "compute_real_reference": COMPUTE_REAL_REFERENCE,
    "vendi_score": importlib.util.find_spec("vendi_score") is not None,
}
display(checks)
ready = (
    len(sample_paths) >= 2
    and (not COMPUTE_REAL_REFERENCE or len(real_paths) >= 2)
    and checks["vendi_score"]
)


## 3. Extract official image embeddings and compute Vendi

Embedding batches are loaded from disk incrementally. Both embedding extraction and `score_dual` come from the official package.


In [ ]:
vendi_generated = None
vendi_real = None


def path_batches(paths: list[Path], batch_size: int):
    for start in range(0, len(paths), batch_size):
        yield paths[start : start + batch_size]


if not RUN_VENDI:
    print("Vendi skipped. Set RUN_VENDI = True after preflight passes.")
elif not ready:
    raise ValueError("Vendi preflight did not pass.")
else:
    import numpy as np
    import torch
    from PIL import Image
    from tqdm.auto import tqdm
    from vendi_score import image_utils, vendi

    device = torch.device(DEVICE)

    def get_vendi_inception_pool_model():
        try:
            return image_utils.get_inception(pretrained=True, pool=True)
        except ValueError as exc:
            if "init_weights" not in str(exc):
                raise
            # vendi_score<=0.0.3 calls torchvision's legacy pretrained=True path
            # with init_weights=True. New torchvision requires init_weights=False
            # when pretrained weights are requested. This fallback builds the same
            # Inception-v3 pool embedding model with torchvision's current API.
            import torch.nn as nn
            from torchvision.models import Inception_V3_Weights, inception_v3

            weights = Inception_V3_Weights.DEFAULT
            model = inception_v3(
                weights=weights,
                transform_input=True,
                init_weights=False,
            ).eval()
            model.fc = nn.Identity()
            return model

    model = get_vendi_inception_pool_model().to(device).eval()
    transform = image_utils.inception_transforms()

    def official_embeddings(paths: list[Path], description: str) -> np.ndarray:
        chunks = []
        total_batches = (len(paths) + BATCH_SIZE - 1) // BATCH_SIZE
        for batch_paths in tqdm(path_batches(paths, BATCH_SIZE), total=total_batches, desc=description):
            images = []
            for path in batch_paths:
                with Image.open(path) as image:
                    images.append(image.convert("RGB").copy())
            chunks.append(
                image_utils.get_embeddings(
                    images,
                    model=model,
                    transform=transform,
                    batch_size=BATCH_SIZE,
                    device=device,
                )
            )
        return np.concatenate(chunks, axis=0)

    started = time.perf_counter()
    sample_embeddings = official_embeddings(sample_paths, "Generated Inception embeddings")
    vendi_generated = float(vendi.score_dual(sample_embeddings, normalize=True))

    if COMPUTE_REAL_REFERENCE:
        real_embeddings = official_embeddings(real_paths, "Real Inception embeddings")
        vendi_real = float(vendi.score_dual(real_embeddings, normalize=True))
    elapsed = time.perf_counter() - started

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    ratio = (
        vendi_generated / vendi_real
        if vendi_real is not None and vendi_real > 0
        else None
    )
    rows = [
        "| Metric | Value |",
        "|---|---:|",
        f"| Generated Vendi | {vendi_generated:.6f} |",
    ]
    if vendi_real is not None:
        rows.extend([
            f"| Real Vendi | {vendi_real:.6f} |",
            f"| Generated / real | {ratio:.6f} |",
        ])
    display(Markdown("\n".join(rows)))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base = OUTPUT_PATH.expanduser().resolve()
    output_path = base.with_name(f"{base.stem}_{timestamp}{base.suffix}")
    payload = {
        "created_at_local": datetime.now().astimezone().isoformat(),
        "metric": "Vendi Score with official Inception-v3 image embeddings",
        "vendi_generated": vendi_generated,
        "vendi_real": vendi_real,
        "vendi_generated_to_real_ratio": ratio,
        "samples_dir": str(SAMPLES_DIR.expanduser().resolve()),
        "real_dir": str(REAL_DIR.expanduser().resolve()) if COMPUTE_REAL_REFERENCE else None,
        "sample_images": len(sample_paths),
        "real_images": len(real_paths),
        "device": str(device),
        "batch_size": BATCH_SIZE,
        "runtime_seconds": elapsed,
        "official_source": "https://github.com/vertaix/Vendi-Score",
    }
    output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    print(f"Saved: {output_path}")


## Notes

- This uses the official package's predefined torchvision Inception-v3 image embedding utility, not DINOv2.
- Vendi measures diversity, not realism. Use CMMD or FD-DINOv2 separately for realism.
- Compare generated Vendi scores using the same embedding model and the same number of images.
